# ROBERT Run Wrapper and Output Archiver

Use this notebook to run ROBERT (optional) and archive `CURATE`, `GENERATE`, `VERIFY`, and `PREDICT` into a unique, timestamped folder per run.

This prevents new runs from overwriting previous generic outputs.

## Step 1: Choose the dataset and ROBERT options

This is the main cell a chemist should edit before running the notebook.

What you need to decide:

1. `DATASET_RELATIVE`: the CSV file you want ROBERT to analyze. You can enter a path from the project root, such as `Databases/Regression/Hvapor.csv`, or just a filename, such as `Hvapor.csv`, and the notebook will search under `Databases/`.
2. `ROBERT_OPTIONS["names"]`: the column that identifies each molecule, compound, reaction, or datapoint. This is often a name, ID, code, or SMILES column.
3. `ROBERT_OPTIONS["y"]`: the target property or outcome ROBERT should model.
4. Optional settings such as `ignore`, `type`, `model`, `csv_test`, `kfold`, or `seed` can be uncommented only when you need them.
5. Keep `RUN_ROBERT = False` while checking the resolved command and previewing the data. Set it to `True` only when you are ready to run ROBERT.

After editing the next cell, run the notebook from top to bottom. Use the preview step below to confirm that the selected columns look correct before executing ROBERT.


In [56]:
# -----------------------------------------------------------------------------
# USER SETUP: edit this cell for each ROBERT run.
# -----------------------------------------------------------------------------

# 1) Choose the CSV file to analyze.
#    Use either a project-relative path or just the filename.
DATASET_RELATIVE = "AQME-ROBERT_interpret_CO2.csv"

# 2) Choose ROBERT command options.
#    At minimum, set:
#    - names: the molecule/datapoint identifier column
#    - y: the target property or outcome column
ROBERT_OPTIONS = {
    "names": "SMILES",
    "y": "MolLogP",
    # "ignore": "[Name]",
    # "type": "reg",
    # "model": "[RF,GB,NN,MVL]",
    # "csv_test": "path/to/external_test.csv",
    # "kfold": 5,
    # "seed": 0,
}

# 3) Choose whether to actually run ROBERT.
#    Recommended first pass: leave False, run the preview, then set True when ready.
RUN_ROBERT = True

# 4) Choose archive behavior.
#    False moves ROBERT output folders into the archive.
#    True copies them and leaves the root-level folders in place.
COPY_INSTEAD_OF_MOVE = False


## Step 2: Resolve the selected dataset

This cell is implementation setup. You normally do not edit it.

It imports the required Python tools, finds the project root, resolves the CSV path from `DATASET_RELATIVE`, and prepares constants used later by the wrapper.


In [57]:
from pathlib import Path
from datetime import datetime
import json
import shutil
import subprocess
from difflib import get_close_matches


def resolve_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / "AGENTS.md").exists() and (candidate / "robert").exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not infer project root from notebook working directory. "
        "Expected a folder containing AGENTS.md and robert/."
    )


def dedupe_paths_case_insensitive(paths: list[Path]) -> list[Path]:
    bucket = {}
    for p in paths:
        key = str(p.resolve()).lower()
        if key not in bucket:
            bucket[key] = p.resolve()
    return list(bucket.values())


def resolve_dataset_path(project_root: Path, dataset_input: str) -> Path:
    candidate = Path(dataset_input)

    # 1) Absolute or direct relative path from project root
    if candidate.is_absolute() and candidate.exists():
        return candidate.resolve()

    requested = (project_root / candidate).resolve()
    if requested.exists():
        return requested

    # 2) Search by filename under Databases/databases
    db_roots_raw = [project_root / "Databases", project_root / "databases"]
    db_roots = dedupe_paths_case_insensitive([p for p in db_roots_raw if p.exists()])

    all_csvs = []
    exact_matches = []
    for db_root in db_roots:
        for csv_path in db_root.rglob("*.csv"):
            resolved_csv = csv_path.resolve()
            all_csvs.append(resolved_csv)
            if csv_path.name.lower() == candidate.name.lower():
                exact_matches.append(resolved_csv)

    all_csvs = dedupe_paths_case_insensitive(all_csvs)
    exact_matches = dedupe_paths_case_insensitive(exact_matches)

    if len(exact_matches) == 1:
        return exact_matches[0]
    if len(exact_matches) > 1:
        shown = "\n".join(str(p) for p in exact_matches[:10])
        raise FileNotFoundError(
            "Multiple datasets matched your filename. Use a more specific DATASET_RELATIVE path.\n"
            f"Matches:\n{shown}"
        )

    # 3) Partial-name fallback (for inputs like vapor.csv -> Hvapor.csv)
    token = candidate.stem.lower()
    partial_matches = [p for p in all_csvs if token and token in p.stem.lower()]
    partial_matches = dedupe_paths_case_insensitive(partial_matches)

    if len(partial_matches) == 1:
        print(f"Using partial match for dataset input '{dataset_input}': {partial_matches[0]}")
        return partial_matches[0]
    if len(partial_matches) > 1:
        shown = "\n".join(str(p) for p in partial_matches[:10])
        raise FileNotFoundError(
            "Multiple partial matches found. Use a more specific DATASET_RELATIVE path.\n"
            f"Matches:\n{shown}"
        )

    # 4) Suggest nearest filenames
    all_names = sorted({p.name for p in all_csvs})
    suggestions = get_close_matches(candidate.name, all_names, n=8, cutoff=0.3)
    suggestion_text = "\n".join(suggestions) if suggestions else "(no close filename suggestions)"

    alt_requested = (project_root / str(candidate).replace("Databases/", "databases/")).resolve()
    raise FileNotFoundError(
        f"Dataset not found. Tried: {requested}, {alt_requested}, and search under Databases/.\n"
        f"Closest filenames:\n{suggestion_text}\n"
        "Update DATASET_RELATIVE in the USER SETUP cell."
    )


PROJECT_ROOT = resolve_project_root()
DATASET_CSV = resolve_dataset_path(PROJECT_ROOT, DATASET_RELATIVE)
RUNS_ROOT = PROJECT_ROOT / "agent" / "run_archive"
OUTPUT_DIR_NAMES = ["CURATE", "GENERATE", "VERIFY", "PREDICT"]

# Command display and execution behavior. These are not normally edited.
COMMAND_OPTION_ORDER = ["names", "y"]
COMMAND_CSV_NAME = DATASET_CSV.name  # display/copy-paste friendly
EXECUTION_CSV_NAME = str(DATASET_CSV)  # robust path for subprocess execution

print("Resolved project root:", PROJECT_ROOT)
print("Resolved dataset:", DATASET_CSV)
print("ROBERT will run:", RUN_ROBERT)


Resolved project root: /Users/cjcscha/ROBERT/helper_rob/robert
Resolved dataset: /Users/cjcscha/ROBERT/helper_rob/robert/Databases/Regression/AQME-ROBERT_interpret_CO2.csv
ROBERT will run: True


## Step 3: Preview dataset columns and sample rows

Run this cell before executing ROBERT.

It helps you identify:
- the molecule/name column,
- the target property column,
- columns that should be ignored,
- candidate descriptor columns.

If the detected columns are not what you expect, go back to the USER SETUP cell and update `ROBERT_OPTIONS`.


In [58]:
import pandas as pd

preview_rows = 8
df_preview = pd.read_csv(DATASET_CSV, nrows=preview_rows)

print(f"Dataset file: {DATASET_CSV}")
print(f"Preview rows loaded: {len(df_preview)}")
print(f"Column count: {len(df_preview.columns)}")

columns = list(df_preview.columns)
print("\nColumns:")
for idx, col in enumerate(columns, start=1):
    print(f"{idx:>3}. {col}")

name_hint_keywords = ["name", "mol", "molecule", "code", "id", "smiles"]
target_hint_keywords = ["target", "y", "yield", "ee", "ddg", "dg", "barrier", "tof", "activity", "log", "k", "outcome", "class"]

name_hints = [c for c in columns if any(k in c.lower() for k in name_hint_keywords)]
target_hints = [c for c in columns if any(k in c.lower() for k in target_hint_keywords)]

print("\nLikely name/molecule columns:", name_hints if name_hints else "none detected")
print("Likely target (y) columns:", target_hints if target_hints else "none detected")

ignore_suggestions = []
for col in name_hints:
    if col not in ignore_suggestions:
        ignore_suggestions.append(col)
if "smiles" in [c.lower() for c in columns]:
    smiles_exact = [c for c in columns if c.lower() == "smiles"]
    for col in smiles_exact:
        if col not in ignore_suggestions:
            ignore_suggestions.append(col)

feature_candidates = [c for c in columns if c not in set(ignore_suggestions + target_hints)]
print("\nCandidate ignore columns:", ignore_suggestions if ignore_suggestions else "review manually")
print(f"Candidate X-feature count after quick filtering: {len(feature_candidates)}")

print("\nTop preview rows:")
display(df_preview.head(preview_rows))

print("\nNext step:")
print("Update ROBERT_OPTIONS in the USER SETUP cell using the inspected column names.")

Dataset file: /Users/cjcscha/ROBERT/helper_rob/robert/Databases/Regression/AQME-ROBERT_interpret_CO2.csv
Preview rows loaded: 8
Column count: 42

Columns:
  1. code_name
  2. SMILES
  3. g_barr
  4. O1CC1_O_Partial charge
  5. O1CC1_O_Dipole moment
  6. O1CC1_O_Electrophil.
  7. O1CC1_O_Nucleophil.
  8. O1CC1_O_Radical attack
  9. O1CC1_O_SASA
 10. O1CC1_O_Buried volume
 11. O1CC1_O_H bond with H2O
 12. O1CC1_O_s proportion
 13. O1CC1_O_p proportion
 14. O1CC1_O_d proportion
 15. O1CC1_O_Coord. numbers
 16. O1CC1_O_Polariz. alpha
 17. O1CC1_O_FOD
 18. O1CC1_O_Dispersion
 19. O1CC1_O_Pyramidalization
 20. O1CC1_O_Pyramidaliz. volume
 21. HOMO-LUMO gap
 22. HOMO
 23. LUMO
 24. IP
 25. EA
 26. Dipole module
 27. Total charge
 28. Global SASA
 29. G solv. in H2O
 30. G of H-bonds H2O
 31. Fermi-level
 32. Total polariz. alpha
 33. Total FOD
 34. Hardness
 35. Softness
 36. Electronegativity
 37. Electrophil. idx
 38. Nucleophilicity idx
 39. Second IP
 40. Second EA
 41. S0-T1 gap
 42. Mol

,code_name,SMILES,g_barr,O1CC1_O_Partial charge,O1CC1_O_Dipole moment,O1CC1_O_Electrophil.,O1CC1_O_Nucleophil.,O1CC1_O_Radical attack,O1CC1_O_SASA,O1CC1_O_Buried volume,...,Total FOD,Hardness,Softness,Electronegativity,Electrophil. idx,Nucleophilicity idx,Second IP,Second EA,S0-T1 gap,MolLogP
0,H,O1CC1,21.21,-0.0601,0.6849,0.0270,0.0203,0.0339,40.4485,0.2804,...,0.0001,17.3250,0.0577,2.0328,0.1193,2.1388,18.5491,-5.1842,220.5947,0.0166
1,Me,CC1OC1,23.11,-0.0632,0.6659,0.0199,0.0145,0.0219,36.7986,0.3360,...,0.0001,16.5961,0.0603,1.7955,0.0971,2.3482,16.7405,-4.3453,218.1553,0.4051
2,CF3,FC(C1OC1)(F)F,25.44,-0.0285,0.6673,0.0702,0.0474,0.0828,34.0687,0.3762,...,0.0002,16.1438,0.0619,3.3662,0.3509,1.4732,18.1931,-2.4492,208.0128,0.9475
3,tBu,CC(C)(C)C1OC1,26.17,-0.0510,0.6711,0.0130,0.0057,0.0127,28.7489,0.4050,...,0.0002,15.5455,0.0643,1.4816,0.0706,2.6975,14.6320,-3.2362,214.2380,1.4313
4,OH,OC1OC1,22.96,-0.0597,0.6541,0.0262,0.0150,0.0268,37.3486,0.3198,...,0.0001,17.0542,0.0586,2.0378,0.1217,2.1475,17.9972,-4.8537,217.4912,-0.6650
5,Ph,C1(C2=CC=CC=C2)OC1,24.61,-0.0499,0.6545,0.0358,0.0606,0.0593,31.4388,0.3762,...,0.0386,10.3705,0.0964,3.6751,0.6512,1.3199,14.2496,2.3656,106.1888,1.7579
6,C(CF3)3,FC(C(C(F)(F)F)(C(F)(F)F)C1OC1)(F)F,32.72,-0.0234,0.6734,0.0798,0.0782,0.1017,24.6491,0.4800,...,0.0002,15.2876,0.0654,4.0883,0.5467,1.2216,17.2691,0.1159,206.9688,3.0585
7,CH2-Br,BrCC1OC1,23.95,-0.0409,0.6728,0.0634,0.0653,0.1008,36.4591,0.3438,...,0.0164,11.5705,0.0864,4.0602,0.7124,1.1968,15.9389,1.6783,123.0374,0.7801



Next step:
Update ROBERT_OPTIONS in the USER SETUP cell using the inspected column names.


## Step 4: Wrapper helper functions

This cell defines utility functions used by the run workflow.

You normally do not edit this cell. The functions below create the run archive folder, build the ROBERT command, run ROBERT when requested, archive the output folders, and write `run_manifest.json` for traceability.


In [59]:
def sanitize_name(text: str) -> str:
    cleaned = []
    for ch in text:
        if ch.isalnum() or ch in ['-', '_']:
            cleaned.append(ch)
        else:
            cleaned.append('_')
    return ''.join(cleaned).strip('_') or 'dataset'


def build_run_folder(runs_root: Path, dataset_csv: Path) -> Path:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    dataset_tag = sanitize_name(dataset_csv.stem)
    run_folder = runs_root / f"{timestamp}__{dataset_tag}"
    run_folder.mkdir(parents=True, exist_ok=False)
    return run_folder


def options_to_cli_args(options: dict, preferred_order: list[str] | None = None) -> list[str]:
    args = []
    ordered_keys = []
    seen = set()

    if preferred_order:
        for key in preferred_order:
            if key in options:
                ordered_keys.append(key)
                seen.add(key)

    for key in options:
        if key not in seen:
            ordered_keys.append(key)

    for key in ordered_keys:
        value = options[key]
        flag = f"--{key}"
        if isinstance(value, bool):
            if value:
                args.append(flag)
            continue
        if value is None:
            continue
        args.extend([flag, str(value)])
    return args


def build_robert_command(csv_name_arg: str, options: dict, preferred_order: list[str] | None = None) -> list[str]:
    command = ["python", "-m", "robert"]
    command.extend(options_to_cli_args(options, preferred_order=preferred_order))
    command.extend(["--csv_name", str(csv_name_arg)])
    return command


def run_robert_if_requested(command: list[str], project_root: Path, enabled: bool) -> int | None:
    if not enabled:
        print("RUN_ROBERT is False: skipping ROBERT execution and archiving existing output folders only.")
        return None

    print("Running ROBERT command:")
    print(' '.join(command))
    result = subprocess.run(command, cwd=project_root, check=False)
    print(f"ROBERT exit code: {result.returncode}")
    return result.returncode


def archive_output_dirs(project_root: Path, run_folder: Path, output_dir_names: list[str], copy_only: bool) -> list[str]:
    archive_root = run_folder / "outputs"
    archive_root.mkdir(parents=True, exist_ok=True)

    archived = []
    for name in output_dir_names:
        src = project_root / name
        if not src.exists() or not src.is_dir():
            continue

        dest = archive_root / name
        if dest.exists():
            suffix = datetime.now().strftime("%H%M%S")
            dest = archive_root / f"{name}_{suffix}"

        if copy_only:
            shutil.copytree(src, dest)
        else:
            shutil.move(str(src), str(dest))

        archived.append(name)

    return archived


def detect_robert_version_from_archived_outputs(run_folder: Path) -> str | None:
    """Read ROBERT version from archived .dat headers if available."""
    dat_candidates = [
        run_folder / "outputs" / "PREDICT" / "PREDICT_data.dat",
        run_folder / "outputs" / "VERIFY" / "VERIFY_data.dat",
        run_folder / "outputs" / "GENERATE" / "GENERATE_data.dat",
        run_folder / "outputs" / "CURATE" / "CURATE_data.dat",
    ]
    for dat in dat_candidates:
        if not dat.exists():
            continue
        try:
            lines = dat.read_text(encoding="utf-8").splitlines()
        except Exception:
            continue
        for line in lines[:10]:
            if "ROBERT v" in line:
                tail = line.split("ROBERT v", 1)[-1].strip()
                return tail.split()[0] if tail else None
    return None


def find_report_pdfs(project_root: Path, run_folder: Path) -> list[str]:
    """Collect report PDF paths relevant to this run for backend/UI linkage."""
    candidates = []

    root_pdf = project_root / "ROBERT_report.pdf"
    if root_pdf.exists():
        candidates.append(root_pdf.resolve())

    candidates.extend([p.resolve() for p in run_folder.glob("**/*.pdf") if p.is_file()])

    seen = set()
    unique = []
    for p in candidates:
        s = str(p)
        if s not in seen:
            seen.add(s)
            unique.append(s)
    return unique


def write_manifest(run_folder: Path, project_root: Path, dataset_csv: Path, archived_dirs: list[str],
                   command: list[str], run_robert: bool, return_code: int | None, copy_only: bool) -> Path:
    # The manifest is the run's audit trail. It records enough information to
    # understand which dataset was used, which ROBERT command was built, what
    # folders were archived, and where any report PDFs were found.

    # ROBERT writes its version near the top of module .dat files. After the
    # output folders are archived, read those headers so the run records the
    # ROBERT version used for this result.
    robert_version = detect_robert_version_from_archived_outputs(run_folder)

    # Link any available ROBERT_report.pdf files. The extractor and UI can use
    # this list later to copy/display the report alongside extracted evidence.
    report_pdfs = find_report_pdfs(project_root, run_folder)

    # Store user-facing provenance and execution metadata in one JSON object.
    # This does not affect ROBERT outputs; it only documents this wrapper run.
    manifest = {
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "project_root": str(project_root.resolve()),
        "dataset_csv": str(dataset_csv.resolve()),
        "run_robert": run_robert,
        "robert_command": command,
        "robert_return_code": return_code,
        "robert_version": robert_version,
        "copy_instead_of_move": copy_only,
        "archived_output_dirs": archived_dirs,
        "expected_output_dirs": OUTPUT_DIR_NAMES,
        "report_pdfs": report_pdfs,
    }

    # Write the manifest next to outputs/ inside the run archive folder.
    # Example: agent/run_archive/<timestamp>__<dataset>/run_manifest.json
    manifest_path = run_folder / "run_manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return manifest_path

## Step 5: Execute ROBERT and archive the run

This cell performs the workflow in order.

It will:
1. verify that the selected dataset file exists,
2. build and print the final ROBERT command,
3. optionally run ROBERT,
4. move or copy `CURATE`, `GENERATE`, `VERIFY`, and `PREDICT` into a unique archive folder,
5. write `run_manifest.json` with metadata for reproducibility.

`run_manifest.json` is useful because ROBERT's standard output folders have generic names and can be overwritten by later runs. The manifest preserves the basic history of this archived run so you, a collaborator, or the companion agent can tell exactly what produced the results.

The manifest records:
- when the archive was created,
- the project folder and dataset CSV path,
- the exact ROBERT command built by the notebook,
- whether ROBERT was actually run from this notebook,
- ROBERT's return code when a run was executed,
- the detected ROBERT version when available from `.dat` files,
- whether folders were moved or copied,
- which output folders were archived,
- any linked `ROBERT_report.pdf` files found for the run.

If you only want to preview the command and archive any existing output folders, keep `RUN_ROBERT = False` in the USER SETUP cell.


In [60]:
if not DATASET_CSV.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATASET_CSV}. "
        "Update DATASET_RELATIVE in the USER SETUP cell and rerun the setup cells."
    )

ROBERT_COMMAND_DISPLAY = build_robert_command(
    COMMAND_CSV_NAME, ROBERT_OPTIONS, preferred_order=COMMAND_OPTION_ORDER
    )
ROBERT_COMMAND = build_robert_command(
    EXECUTION_CSV_NAME, ROBERT_OPTIONS, preferred_order=COMMAND_OPTION_ORDER
    )

print("Resolved ROBERT command (copy/paste style):", ' '.join(ROBERT_COMMAND_DISPLAY))
print("Execution ROBERT command:", ' '.join(ROBERT_COMMAND))

RUNS_ROOT.mkdir(parents=True, exist_ok=True)
run_folder = build_run_folder(RUNS_ROOT, DATASET_CSV)

return_code = run_robert_if_requested(ROBERT_COMMAND, PROJECT_ROOT, RUN_ROBERT)
archived_dirs = archive_output_dirs(PROJECT_ROOT, run_folder, OUTPUT_DIR_NAMES, COPY_INSTEAD_OF_MOVE)
manifest_path = write_manifest(
    run_folder=run_folder,
    project_root=PROJECT_ROOT,
    dataset_csv=DATASET_CSV,
    archived_dirs=archived_dirs,
    command=ROBERT_COMMAND,
    run_robert=RUN_ROBERT,
    return_code=return_code,
    copy_only=COPY_INSTEAD_OF_MOVE,
)

print("Run folder:", run_folder)
print("Archived output dirs:", archived_dirs if archived_dirs else "none found")
print("Manifest:", manifest_path)

Resolved ROBERT command (copy/paste style): python -m robert --names SMILES --y MolLogP --csv_name AQME-ROBERT_interpret_CO2.csv
Execution ROBERT command: python -m robert --names SMILES --y MolLogP --csv_name /Users/cjcscha/ROBERT/helper_rob/robert/Databases/Regression/AQME-ROBERT_interpret_CO2.csv
Running ROBERT command:
python -m robert --names SMILES --y MolLogP --csv_name /Users/cjcscha/ROBERT/helper_rob/robert/Databases/Regression/AQME-ROBERT_interpret_CO2.csv
ROBERT v 2.1.0 2026/05/21 14:11:12 
How to cite: Dalmau, D.; Alegre Requena, J. V. WIREs Comput Mol Sci. 2024, 14, e1733.


Command line used in ROBERT: python -m robert --names "SMILES" --y "MolLogP" --csv_name "/Users/cjcscha/ROBERT/helper_rob/robert/Databases/Regression/AQME-ROBERT_interpret_CO2.csv"



o  Starting data curation with the CURATE module


o  Database AQME-ROBERT_interpret_CO2.csv loaded successfully, including:
   - 19 datapoints
   - 40 accepted descriptors
   - 1 ignored descriptors
   - 0 discarded desc

## Usage Notes

1. Edit only the USER SETUP cell at the top for a normal run.
2. Set `DATASET_RELATIVE` to your CSV path or filename.
3. Set `ROBERT_OPTIONS["names"]` to the molecule/datapoint identifier column.
4. Set `ROBERT_OPTIONS["y"]` to the target property or outcome column.
5. Run the preview cell and confirm the columns look correct.
6. Keep `RUN_ROBERT = False` until the command and preview look right; set it to `True` when ready to execute ROBERT.
7. If you want to keep generic output folders in place, set `COPY_INSTEAD_OF_MOVE = True`.

Each run is archived in `agent/run_archive/<timestamp>__<dataset_name>/`.
